# POI Data Classification

This notebook classifies Points of Interest (POI) into four main categories:
- **Business**: Commercial establishments (shops, restaurants, cafes, pubs)
- **Tourism**: Tourism-related facilities (guest houses, information centers)
- **Public Services**: Government and community services (police, post office, places of worship)
- **Transportation**: Transportation infrastructure (traffic signals, ferry terminals, platforms)

In [ ]:
# Import required libraries
import pandas as pd
import os
import sys

# Add project root to path to ensure imports work correctly
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"Current working directory: {os.getcwd()}")

## 1. Load POI Data

In [ ]:
# Load the POI data from CSV
data_path = os.path.join(project_root, 'data', 'poi_data.csv')
df = pd.read_csv(data_path)

print(f"Loaded {len(df)} POI records")
print(f"\nData shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

## 2. Define Classification Rules

We'll create a classification function based on the POI_type and POI fields.

In [ ]:
def classify_poi(poi_type, poi):
    """
    Classify a POI into one of four categories:
    - business
    - tourism
    - public_services
    - transportation
    
    Parameters:
    -----------
    poi_type : str
        The type category of the POI (e.g., 'amenity', 'tourism', 'shop')
    poi : str
        The specific POI subcategory (e.g., 'restaurant', 'guest_house')
    
    Returns:
    --------
    str
        The classified category
    """
    
    # Business establishments
    business_pois = {
        'shop': ['convenience', 'supermarket', 'bakery', 'butcher', 'clothes'],
        'amenity': ['restaurant', 'cafe', 'pub', 'bar', 'fast_food', 'bank', 'pharmacy']
    }
    
    # Tourism facilities
    tourism_pois = {
        'tourism': ['guest_house', 'hotel', 'motel', 'information', 'museum', 'attraction', 'viewpoint']
    }
    
    # Public services
    public_service_pois = {
        'amenity': ['police', 'post_office', 'social_facility', 'place_of_worship', 
                    'post_box', 'townhall', 'library', 'hospital', 'clinic', 'school', 'university']
    }
    
    # Transportation infrastructure
    transportation_pois = {
        'highway': ['traffic_signals', 'bus_stop', 'crossing'],
        'amenity': ['ferry_terminal', 'bus_station', 'taxi', 'parking'],
        'public_transport': ['platform', 'station', 'stop_position']
    }
    
    # Classification logic with priority
    # 1. Check tourism first (most specific)
    if poi_type in tourism_pois and poi in tourism_pois[poi_type]:
        return 'tourism'
    
    # 2. Check transportation
    if poi_type in transportation_pois and poi in transportation_pois[poi_type]:
        return 'transportation'
    
    # 3. Check public services
    if poi_type in public_service_pois and poi in public_service_pois[poi_type]:
        return 'public_services'
    
    # 4. Check business
    if poi_type in business_pois and poi in business_pois[poi_type]:
        return 'business'
    
    # 5. Default fallback based on poi_type
    if poi_type == 'tourism':
        return 'tourism'
    elif poi_type in ['highway', 'public_transport']:
        return 'transportation'
    elif poi_type == 'shop':
        return 'business'
    else:
        return 'other'

# Test the function with a sample
print("Sample classifications:")
print(f"restaurant -> {classify_poi('amenity', 'restaurant')}")
print(f"guest_house -> {classify_poi('tourism', 'guest_house')}")
print(f"police -> {classify_poi('amenity', 'police')}")
print(f"traffic_signals -> {classify_poi('highway', 'traffic_signals')}")

## 3. Apply Classification to Dataset

In [ ]:
# Apply the classification function to all rows
df['category'] = df.apply(lambda row: classify_poi(row['POI_type'], row['POI']), axis=1)

print("Classification completed!")
print(f"\nDataFrame with classifications:")
df

## 4. Analyze Classification Results

In [ ]:
# Count POIs by category
category_counts = df['category'].value_counts()
print("POI Count by Category:")
print(category_counts)
print(f"\nTotal POIs: {len(df)}")

In [ ]:
# Show percentage distribution
category_percentages = (df['category'].value_counts(normalize=True) * 100).round(2)
print("Category Distribution (%)")
print(category_percentages)

In [ ]:
# Group by category and show examples
print("\nExamples by Category:")
print("=" * 80)
for category in df['category'].unique():
    print(f"\n{category.upper()}:")
    print("-" * 80)
    category_df = df[df['category'] == category][['POI_type', 'POI', 'latitude', 'longitude']]
    print(category_df.to_string(index=False))
    print()

## 5. Save Classified Data

In [ ]:
# Save the classified data to a new CSV file
output_path = os.path.join(project_root, 'data', 'poi_data_classified.csv')
df.to_csv(output_path, index=False)
print(f"Classified data saved to: {output_path}")

## 6. Summary Statistics

In [ ]:
# Create a summary table
summary = df.groupby('category').agg({
    'POI': 'count',
    'latitude': ['min', 'max', 'mean'],
    'longitude': ['min', 'max', 'mean']
}).round(6)

summary.columns = ['Count', 'Min_Lat', 'Max_Lat', 'Avg_Lat', 'Min_Lon', 'Max_Lon', 'Avg_Lon']
print("\nSummary Statistics by Category:")
print(summary)

## 7. Export Classification Function for Reuse

In [ ]:
# Save the classification function to a Python module for reuse
src_path = os.path.join(project_root, 'src', 'poi_classifier.py')

classification_code = '''"""POI Classification Module

This module provides functions to classify Points of Interest (POI) into categories.
"""

def classify_poi(poi_type, poi):
    """
    Classify a POI into one of four categories:
    - business
    - tourism
    - public_services
    - transportation
    
    Parameters:
    -----------
    poi_type : str
        The type category of the POI (e.g., 'amenity', 'tourism', 'shop')
    poi : str
        The specific POI subcategory (e.g., 'restaurant', 'guest_house')
    
    Returns:
    --------
    str
        The classified category
    """
    
    # Business establishments
    business_pois = {
        'shop': ['convenience', 'supermarket', 'bakery', 'butcher', 'clothes'],
        'amenity': ['restaurant', 'cafe', 'pub', 'bar', 'fast_food', 'bank', 'pharmacy']
    }
    
    # Tourism facilities
    tourism_pois = {
        'tourism': ['guest_house', 'hotel', 'motel', 'information', 'museum', 'attraction', 'viewpoint']
    }
    
    # Public services
    public_service_pois = {
        'amenity': ['police', 'post_office', 'social_facility', 'place_of_worship', 
                    'post_box', 'townhall', 'library', 'hospital', 'clinic', 'school', 'university']
    }
    
    # Transportation infrastructure
    transportation_pois = {
        'highway': ['traffic_signals', 'bus_stop', 'crossing'],
        'amenity': ['ferry_terminal', 'bus_station', 'taxi', 'parking'],
        'public_transport': ['platform', 'station', 'stop_position']
    }
    
    # Classification logic with priority
    if poi_type in tourism_pois and poi in tourism_pois[poi_type]:
        return 'tourism'
    
    if poi_type in transportation_pois and poi in transportation_pois[poi_type]:
        return 'transportation'
    
    if poi_type in public_service_pois and poi in public_service_pois[poi_type]:
        return 'public_services'
    
    if poi_type in business_pois and poi in business_pois[poi_type]:
        return 'business'
    
    # Default fallback based on poi_type
    if poi_type == 'tourism':
        return 'tourism'
    elif poi_type in ['highway', 'public_transport']:
        return 'transportation'
    elif poi_type == 'shop':
        return 'business'
    else:
        return 'other'
'''

with open(src_path, 'w') as f:
    f.write(classification_code)

print(f"Classification function saved to: {src_path}")
print("You can now import it using: from src.poi_classifier import classify_poi")